# Ymmo — Analyse du marché immobilier

**Objectifs (cf. cahier des charges)**
- Nettoyer et préparer les données issues de la plateforme
- Identifier les **tendances du marché** par ville et par type de bien
- Repérer les **biens populaires** (vues, demandes)
- Construire un modèle simple de **prédiction de prix**
- Identifier les **zones d'achat intéressantes** (ratio prix/m² vs activité)

**Pré-requis** : avoir lancé `python manage.py seed_demo` pour disposer de données.

**Stack** : pandas, numpy, matplotlib, seaborn, scikit-learn.

## 1. Chargement des données depuis Django

In [ ]:
import os, sys, django
from pathlib import Path

# Configure l'env Django pour pouvoir interroger l'ORM depuis le notebook
BASE_DIR = Path.cwd().parent
sys.path.insert(0, str(BASE_DIR))
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'config.settings')
django.setup()

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
pd.set_option('display.max_columns', 50)

In [ ]:
from apps.properties.models import Property, Agency
from apps.transactions.models import Transaction, VisitRequest

# Charger les biens en DataFrame
qs = Property.objects.values(
    'id', 'reference', 'title', 'property_type', 'transaction_type', 'status',
    'surface', 'rooms', 'bedrooms', 'price', 'energy_class',
    'has_garage', 'has_garden', 'has_pool', 'construction_year',
    'city', 'postal_code', 'region', 'views_count',
    'created_at', 'published_at', 'agency__name',
)
df = pd.DataFrame.from_records(qs)
df['price'] = df['price'].astype(float)
df['price_per_sqm'] = df['price'] / df['surface'].replace(0, np.nan)
print(f'{len(df)} biens chargés')
df.head()

## 2. Nettoyage et préparation

In [ ]:
# Vérifier les valeurs manquantes
print('Valeurs manquantes :')
print(df.isnull().sum()[df.isnull().sum() > 0])

# On ne garde que les biens à la vente, statut publié, sans prix aberrants
active = df[
    (df['status'] == 'AVAILABLE')
    & (df['transaction_type'] == 'SALE')
    & (df['price'].between(20_000, 5_000_000))
    & (df['surface'] > 0)
].copy()

print(f'\n{len(active)} biens actifs après filtrage')
active.describe()[['surface', 'price', 'price_per_sqm', 'views_count']]

## 3. Tendances par ville

In [ ]:
by_city = active.groupby('city').agg(
    nb_listings=('id', 'count'),
    avg_price=('price', 'mean'),
    median_price=('price', 'median'),
    avg_price_sqm=('price_per_sqm', 'mean'),
    avg_surface=('surface', 'mean'),
).sort_values('nb_listings', ascending=False)

by_city.head(15).round(0)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

top10 = by_city.head(10)
top10['nb_listings'].plot(kind='barh', ax=ax[0], color='#1e40af')
ax[0].set_title('Top 10 villes — nombre d\'annonces')
ax[0].set_xlabel('Annonces actives')

top10['avg_price_sqm'].plot(kind='barh', ax=ax[1], color='#f59e0b')
ax[1].set_title('Top 10 villes — prix moyen au m² (€)')
ax[1].set_xlabel('€ / m²')

plt.tight_layout()
plt.show()

## 4. Répartition par type de bien

In [ ]:
by_type = active.groupby('property_type').agg(
    nb=('id', 'count'),
    avg_price=('price', 'mean'),
    avg_surface=('surface', 'mean'),
    avg_price_sqm=('price_per_sqm', 'mean'),
).round(0)
by_type

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

by_type['nb'].plot(kind='pie', ax=ax[0], autopct='%1.1f%%', startangle=90,
                   colors=['#1e40af', '#f59e0b', '#10b981', '#ef4444', '#8b5cf6'])
ax[0].set_title('Répartition par type')
ax[0].set_ylabel('')

sns.boxplot(data=active, x='property_type', y='price_per_sqm', ax=ax[1])
ax[1].set_title('Distribution prix au m² par type')
ax[1].set_yscale('log')
ax[1].set_xlabel('')
ax[1].set_ylabel('€ / m² (log)')

plt.tight_layout()
plt.show()

## 5. Biens populaires (vues)

Les biens les plus consultés indiquent une demande forte — utile pour orienter les agences sur les profils de bien à acquérir.

In [ ]:
popular = active.nlargest(10, 'views_count')[
    ['reference', 'title', 'city', 'property_type', 'surface', 'price', 'views_count']
]
popular

In [ ]:
# Quelles caractéristiques attirent le plus ?
# On segmente par tranche de vues et on regarde les moyennes
active['popularity'] = pd.qcut(active['views_count'], q=4,
                                labels=['Faible', 'Moyen', 'Bon', 'Très populaire'],
                                duplicates='drop')

popularity_profile = active.groupby('popularity', observed=True).agg(
    avg_price=('price', 'mean'),
    avg_surface=('surface', 'mean'),
    avg_rooms=('rooms', 'mean'),
    pct_garage=('has_garage', 'mean'),
    pct_garden=('has_garden', 'mean'),
    pct_pool=('has_pool', 'mean'),
).round(2)
popularity_profile

## 6. Modèle de prédiction de prix

Régression linéaire pour prédire le prix d'un bien à partir de ses caractéristiques.
Permet à un agent d'estimer un nouveau bien rapidement.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score

# Features
features_num = ['surface', 'rooms', 'bedrooms', 'has_garage', 'has_garden', 'has_pool']
features_cat = ['property_type', 'city', 'energy_class']
target = 'price'

data = active.dropna(subset=features_num + features_cat + [target]).copy()
for col in ['has_garage', 'has_garden', 'has_pool']:
    data[col] = data[col].astype(int)

X = data[features_num + features_cat]
y = data[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), features_cat),
], remainder='passthrough')

In [ ]:
results = {}

for name, model in {
    'Régression linéaire': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
}.items():
    pipe = Pipeline([('prep', preprocessor), ('model', model)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    results[name] = {
        'MAE (€)': mean_absolute_error(y_test, pred),
        'R²': r2_score(y_test, pred),
    }

pd.DataFrame(results).T.round({'MAE (€)': 0, 'R²': 3})

In [ ]:
# Visualisation des prédictions vs réel
best_model = Pipeline([('prep', preprocessor),
                       ('model', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))])
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(y_test, y_pred, alpha=0.5, color='#1e40af')
lims = [0, max(y_test.max(), y_pred.max())]
ax.plot(lims, lims, 'r--', label='Prédiction parfaite')
ax.set_xlabel('Prix réel (€)')
ax.set_ylabel('Prix prédit (€)')
ax.set_title('Random Forest — Prédiction vs Réel')
ax.legend()
plt.show()

### Exemple d'estimation pour un nouveau bien

In [ ]:
nouveau_bien = pd.DataFrame([{
    'surface': 85, 'rooms': 4, 'bedrooms': 2,
    'has_garage': 1, 'has_garden': 0, 'has_pool': 0,
    'property_type': 'APARTMENT',
    'city': 'Lyon',
    'energy_class': 'C',
}])
estimation = best_model.predict(nouveau_bien)[0]
print(f'Prix estimé : {estimation:,.0f} €')

## 7. Identification des zones intéressantes pour l'achat

Critère : ville avec **prix au m² inférieur à la médiane nationale** et **forte demande** (mesurée par les vues moyennes par bien).

In [ ]:
median_sqm = active['price_per_sqm'].median()

city_stats = active.groupby('city').agg(
    nb_listings=('id', 'count'),
    avg_price_sqm=('price_per_sqm', 'mean'),
    avg_views=('views_count', 'mean'),
)
city_stats['attractive'] = (
    (city_stats['avg_price_sqm'] < median_sqm)
    & (city_stats['avg_views'] > city_stats['avg_views'].median())
)

# Score : forte demande / faible prix = bonne opportunité
city_stats['opportunity_score'] = (
    city_stats['avg_views'] / city_stats['avg_price_sqm'] * 1000
).round(2)

opportunites = city_stats.sort_values('opportunity_score', ascending=False).head(10)
opportunites.round(0)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(
    city_stats['avg_price_sqm'], city_stats['avg_views'],
    s=city_stats['nb_listings'] * 5,
    c=city_stats['opportunity_score'], cmap='viridis', alpha=0.7,
)
for city, row in city_stats.iterrows():
    ax.annotate(city, (row['avg_price_sqm'], row['avg_views']), fontsize=8)
ax.set_xlabel('Prix moyen au m² (€)')
ax.set_ylabel('Vues moyennes par bien')
ax.set_title('Cartographie des opportunités d\'achat (taille = nb annonces, couleur = score)')
plt.colorbar(scatter, label='Score opportunité')
ax.axvline(median_sqm, color='red', linestyle='--', alpha=0.4, label='Médiane prix/m²')
ax.legend()
plt.tight_layout()
plt.show()

## 8. Synthèse et recommandations

**Pour la direction Ymmo** :
1. Concentrer l'effort commercial sur les villes du top 10 (volume).
2. Cibler les biens populaires (caractéristiques identifiées dans le profil) pour de futures acquisitions.
3. Les zones « attractives » identifiées (prix bas + forte demande) sont des opportunités de prospection.
4. Le modèle Random Forest atteint un R² satisfaisant et peut être utilisé comme outil d'aide à l'estimation pour les agents.

**Pistes d'amélioration** :
- Intégrer des données externes (DVF de l'État pour les prix de transaction réels).
- Ajouter une dimension temporelle pour suivre l'évolution du marché.
- Géolocalisation fine (quartier > ville).